# Phase 1: Domain Research + EDA + Baseline — Visual Product Search Engine
**Date:** 2026-04-20  
**Researcher:** Anthony Rodrigues  

## Research Question
Can we build a visual product search system that retrieves similar fashion products from images? What baseline retrieval quality can a pretrained CNN achieve, and how does it compare to published benchmarks?

## Domain Context
Visual product search is a $40B+ market growing at 17-18% CAGR. Google Lens processes 12B visual searches/month (2025), Pinterest Lens handles 600M queries/month. The task: given a query image of a product, retrieve the most visually similar products from a catalog.

### Published Benchmarks (DeepFashion In-Shop Retrieval)
| Method | Recall@1 | Recall@10 | Recall@20 | Year |
|--------|----------|-----------|-----------|------|
| FashionNet | 53.0 | 73.0 | 76.4 | 2016 |
| HDC (Wang et al.) | 62.1 | 84.9 | 89.0 | 2017 |
| CLIP ViT-B/32 | ~78 | ~93 | ~95 | 2021 |
| DINOv2 ViT-B/14 | ~82 | ~95 | ~97 | 2023 |

### Primary Metric: Recall@K
Standard in image retrieval literature. Recall@1 = fraction of queries where the top-1 result is the correct product. Recall@10 = fraction where correct product appears in top 10. We also track MAP@R (mean average precision at R relevant items).

**Why Recall@K:** The DeepFashion In-Shop benchmark, Stanford Online Products, and virtually all retrieval papers report Recall@K. It directly measures what users care about: "did the correct product appear in the top results?"

### Industry Approach
Amazon StyleSnap uses lightweight CNNs → vector embeddings → ANN search. Pinterest uses visual + text fusion. Both systems rely on learned embeddings indexed with approximate nearest neighbor search (FAISS/HNSW).

## 1. Setup & Data Loading

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RESULTS = PROJECT_ROOT / 'results'

for d in [DATA_RAW, DATA_PROCESSED, RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print('Setup complete.')

Project root: /Users/anthonyrodrigues/Desktop/YC-Portfolio-Projects
Setup complete.


### 1.1 Download Dataset Metadata
We use the Marqo/deepfashion-inshop dataset from HuggingFace. 52,591 images of 12,995 fashion products with rich metadata (gender, category, color, descriptions).

In [2]:
metadata_path = DATA_PROCESSED / 'metadata.csv'

if metadata_path.exists():
    print('Loading cached metadata...')
    df = pd.read_csv(metadata_path)
else:
    print('Downloading metadata from HuggingFace...')
    from datasets import load_dataset
    ds = load_dataset('Marqo/deepfashion-inshop', split='data', streaming=True)
    
    records = []
    for i, ex in enumerate(tqdm(ds, desc='Loading')):
        item_id = ex['item_ID']
        parts = item_id.rsplit('_', 2)
        product_id = parts[0] if len(parts) >= 3 else item_id
        
        records.append({
            'index': i,
            'item_id': item_id,
            'product_id': product_id,
            'category1': ex['category1'],
            'category2': ex['category2'],
            'category3': ex['category3'],
            'color': ex['color'],
            'description': ex['text'] if ex['text'] else '',
        })
    
    df = pd.DataFrame(records)
    df.to_csv(metadata_path, index=False)
    print(f'Saved metadata to {metadata_path}')

print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Loading: 0it [00:00, ?it/s]

Loading: 1it [00:38, 38.11s/it]

Loading: 173it [00:38,  6.45it/s]

Loading: 345it [00:38, 15.54it/s]

Loading: 512it [00:38, 27.98it/s]

...

Loading: 52291it [03:10, 1576.18it/s]

Loading: 52450it [03:10, 1571.43it/s]

Loading: 52591it [03:10, 276.46it/s] 

Saved metadata to /Users/anthonyrodrigues/Desktop/YC-Portfolio-Projects/data/processed/metadata.csv

Dataset shape: (52591, 8)
Columns: ['index', 'item_id', 'product_id', 'category1', 'category2', 'category3', 'color', 'description']


,index,item_id,product_id,category1,category2,category3,color,description
0,0,MEN_Denim_id_00000080_01_1_front,MEN_Denim_id_00000080_01,men,denim,NaN,Black,Give your trusty blues the day off. In a clean...
1,1,MEN_Denim_id_00000080_01_2_side,MEN_Denim_id_00000080_01,men,denim,NaN,Black,Give your trusty blues the day off. In a clean...
2,2,MEN_Denim_id_00000080_01_3_back,MEN_Denim_id_00000080_01,men,denim,NaN,Black,Give your trusty blues the day off. In a clean...
3,3,MEN_Denim_id_00000080_01_7_additional,MEN_Denim_id_00000080_01,men,denim,NaN,Black,Give your trusty blues the day off. In a clean...
4,4,MEN_Denim_id_00000089_01_1_front,MEN_Denim_id_00000089_01,men,denim,NaN,Brown,"Classic in their clean wash and slim fit, thes..."


## 2. Exploratory Data Analysis

### 2.1 Dataset Overview

In [3]:
n_images = len(df)
n_products = df['product_id'].nunique()
n_categories = df['category2'].nunique()
n_colors = df['color'].nunique()

imgs_per_product = df.groupby('product_id').size()

print('=' * 60)
print('DATASET STATISTICS')
print('=' * 60)
print(f'Total images:           {n_images:,}')
print(f'Unique products:        {n_products:,}')
print(f'Product categories:     {n_categories}')
print(f'Unique colors:          {n_colors}')
print(f'Images per product:     {imgs_per_product.mean():.1f} (mean), '
      f'{imgs_per_product.median():.0f} (median), '
      f'{imgs_per_product.min()}-{imgs_per_product.max()} (range)')
print(f'Missing color:          {df["color"].isna().sum()} ({df["color"].isna().mean()*100:.1f}%)')
print(f'Missing description:    {(df["description"] == "").sum()} ({(df["description"] == "").mean()*100:.1f}%)')
print(f'Missing category3:      {df["category3"].isna().sum()} ({df["category3"].isna().mean()*100:.1f}%)')

DATASET STATISTICS
Total images:           52,591
Unique products:        12,995
Product categories:     16
Unique colors:          804
Images per product:     4.0 (mean), 4 (median), 1-6 (range)
Missing color:          0 (0.0%)
Missing description:    0 (0.0%)
Missing category3:      22587 (42.9%)


### 2.2 Category & Gender Distribution

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gender distribution
gender_counts = df['category1'].value_counts()
axes[0].bar(gender_counts.index, gender_counts.values, color=['#e74c3c', '#3498db'])
axes[0].set_title('Gender Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Images')
for i, (cat, val) in enumerate(gender_counts.items()):
    axes[0].text(i, val + 500, f'{val:,}\n({val/n_images*100:.1f}%)', 
                ha='center', fontsize=11, fontweight='bold')

# Category distribution
cat_counts = df['category2'].value_counts()
bars = axes[1].barh(cat_counts.index[::-1], cat_counts.values[::-1])
axes[1].set_title('Product Category Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Images')
for bar, val in zip(bars, cat_counts.values[::-1]):
    axes[1].text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2, 
                f'{val:,}', va='center', fontsize=9)

# Images per product histogram
axes[2].hist(imgs_per_product.values, bins=range(1, imgs_per_product.max()+2), 
            color='#2ecc71', edgecolor='black', alpha=0.7)
axes[2].set_title('Images per Product', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Number of Images')
axes[2].set_ylabel('Number of Products')
axes[2].axvline(imgs_per_product.mean(), color='red', linestyle='--', 
               label=f'Mean={imgs_per_product.mean():.1f}')
axes[2].legend()

plt.tight_layout()
plt.savefig(RESULTS / 'eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: results/eda_distributions.png')

Saved: results/eda_distributions.png


### 2.3 Color Distribution Analysis

In [5]:
color_counts = df['color'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(color_counts.index[::-1], color_counts.values[::-1], color='#9b59b6')
ax.set_title('Top 20 Colors in Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Images')
for bar, val in zip(bars, color_counts.values[::-1]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
           f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS / 'eda_colors.png', dpi=150, bbox_inches='tight')
plt.show()

# Color vs category heatmap
top_colors = df['color'].value_counts().head(10).index
top_cats = df['category2'].value_counts().head(8).index
cross = pd.crosstab(df[df['color'].isin(top_colors)]['color'],
                    df[df['category2'].isin(top_cats)]['category2'])
cross = cross.reindex(index=top_colors, columns=top_cats).fillna(0)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cross, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Color x Category Heatmap (Top 10 Colors x Top 8 Categories)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'eda_color_category_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: results/eda_colors.png, results/eda_color_category_heatmap.png')

Saved: results/eda_colors.png, results/eda_color_category_heatmap.png


### 2.4 Product View Analysis
Each product has multiple views (front, side, back, full, etc.). Understanding the view distribution helps design the retrieval task.

In [6]:
# Extract view type from item_id
def extract_view(item_id):
    parts = item_id.split('_')
    if len(parts) >= 2:
        return parts[-1]  # e.g., 'front', 'side', 'back', 'full'
    return 'unknown'

df['view'] = df['item_id'].apply(extract_view)
view_counts = df['view'].value_counts()

print('View distribution:')
for view, count in view_counts.items():
    print(f'  {view:15s}: {count:6,} ({count/n_images*100:.1f}%)')

print(f'\nUnique views: {df["view"].nunique()}')

# How many products have all standard views?
standard_views = {'front', 'side', 'back'}
views_per_product = df.groupby('product_id')['view'].apply(set)
has_all_standard = views_per_product.apply(lambda x: standard_views.issubset(x)).mean()
print(f'Products with front+side+back: {has_all_standard*100:.1f}%')

View distribution:
  front          : 12,827 (24.4%)
  additional     : 10,914 (20.8%)
  side           : 10,889 (20.7%)
  back           : 10,642 (20.2%)
  full           :  6,636 (12.6%)
  flat           :    683 (1.3%)

Unique views: 6


Products with front+side+back: 66.8%


### 2.5 Description Text Analysis

In [7]:
desc_lengths = df['description'].str.len()
desc_words = df['description'].str.split().str.len()

print('Description statistics:')
print(f'  Mean length (chars): {desc_lengths.mean():.0f}')
print(f'  Mean length (words): {desc_words.mean():.0f}')
print(f'  Empty descriptions:  {(df["description"] == "").sum()}')
print(f'  Longest description: {desc_lengths.max()} chars')

# Sample descriptions
print('\nSample descriptions:')
for _, row in df[df['description'].str.len() > 50].sample(3, random_state=42).iterrows():
    print(f'  [{row["category2"]}] {row["description"][:150]}...')

Description statistics:
  Mean length (chars): 427
  Mean length (words): 73
  Empty descriptions:  0
  Longest description: 997 chars

Sample descriptions:
  [shorts] Give your wardrobe a punch of personality with these shorts! They're boasting a fun tribal-inspired print with comfortable details such as an elastici...
  [blouses] Cut from a silky fabrication and decked out with an eye-catching tie-dye sunburst print, this 3/4-sleeved dolman top adds a visual punch to boyfriend ...
  [sweaters] With its micro-waffle knit structure and classic ribbed trim, this long-sleeved sweater is a prep-school classic - almost. We added eye-catching beadi...


### 2.6 Retrieval Split Creation
Split into gallery (1 image per product) and query (remaining views) for evaluation.

In [8]:
from src.data_pipeline import create_retrieval_splits

train_df, gallery_df, query_df = create_retrieval_splits(df, test_frac=0.2, seed=42)

print('Retrieval split statistics:')
print(f'  Train products:   {train_df["product_id"].nunique():,} ({len(train_df):,} images)')
print(f'  Gallery (test):   {len(gallery_df):,} (1 per test product)')
print(f'  Query (test):     {len(query_df):,} (remaining test views)')
print(f'  Test products:    {gallery_df["product_id"].nunique():,}')

# Save splits
train_df.to_csv(DATA_PROCESSED / 'train.csv', index=False)
gallery_df.to_csv(DATA_PROCESSED / 'gallery.csv', index=False)
query_df.to_csv(DATA_PROCESSED / 'query.csv', index=False)
print('\nSaved splits to data/processed/')

Retrieval split statistics:
  Train products:   10,396 (42,053 images)
  Gallery (test):   2,599 (1 per test product)
  Query (test):     7,939 (remaining test views)
  Test products:    2,599



Saved splits to data/processed/


## 3. Sample Images
Download a small sample of images to visualize products across categories.

In [9]:
# Download sample images for visualization (24 diverse products)
from datasets import load_dataset

sample_indices = []
for cat in df['category2'].value_counts().head(8).index:
    cat_df = df[df['category2'] == cat]
    sample_products = cat_df['product_id'].drop_duplicates().sample(3, random_state=42)
    for pid in sample_products:
        front = cat_df[(cat_df['product_id'] == pid) & (cat_df['view'] == 'front')]
        if len(front) > 0:
            sample_indices.append(front.iloc[0]['index'])
        else:
            sample_indices.append(cat_df[cat_df['product_id'] == pid].iloc[0]['index'])

sample_indices = sorted([int(x) for x in sample_indices])
print(f'Downloading {len(sample_indices)} sample images...')

ds = load_dataset('Marqo/deepfashion-inshop', split='data', streaming=True)
sample_images = {}
sample_meta = {}
needed = set(sample_indices)

for i, ex in enumerate(tqdm(ds, total=max(needed)+1, desc='Fetching samples')):
    if i in needed:
        sample_images[i] = ex['image']
        sample_meta[i] = {'category2': ex['category2'], 'color': ex['color'], 'item_id': ex['item_ID']}
        needed.discard(i)
        if not needed:
            break

print(f'Downloaded {len(sample_images)} images')

Fetching samples:   0%|          | 0/52114 [00:00<?, ?it/s]

Fetching samples:   0%|          | 1/52114 [00:20<302:07:40, 20.87s/it]

Fetching samples:   0%|          | 155/52114 [00:20<1:22:20, 10.52it/s]

Fetching samples:   1%|          | 310/52114 [00:21<34:05, 25.32it/s]  

Fetching samples:   1%|          | 467/52114 [00:21<18:36, 46.26it/s]

...

Fetching samples:  99%|█████████▉| 51644/52114 [01:45<00:00, 1525.64it/s]

Fetching samples:  99%|█████████▉| 51800/52114 [01:45<00:00, 1533.69it/s]

Fetching samples: 100%|█████████▉| 51954/52114 [01:45<00:00, 1529.92it/s]

Fetching samples: 100%|█████████▉| 52113/52114 [01:45<00:00, 494.10it/s] 

Downloaded 24 images


In [10]:
# Visualize sample products grid
n_show = min(24, len(sample_images))
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
fig.suptitle('Sample Products Across Categories', fontsize=16, fontweight='bold', y=1.02)

for ax_idx, (img_idx, img) in enumerate(list(sample_images.items())[:n_show]):
    row, col = divmod(ax_idx, 6)
    ax = axes[row][col]
    ax.imshow(img)
    meta = sample_meta[img_idx]
    ax.set_title(f"{meta['category2']}\n{meta['color']}", fontsize=9)
    ax.axis('off')

# Hide unused axes
for ax_idx in range(n_show, 24):
    row, col = divmod(ax_idx, 6)
    axes[row][col].axis('off')

plt.tight_layout()
plt.savefig(RESULTS / 'sample_products.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/sample_products.png')

Saved: results/sample_products.png


## 4. Baseline Retrieval System

### Hypothesis
A pretrained ResNet50 (ImageNet weights) should provide a reasonable baseline for fashion retrieval. Published baselines using pretrained CNN features typically achieve Recall@1 around 30-50% on DeepFashion without fine-tuning.

### Approach
1. Extract 2048-dim features from ResNet50 (avg pool layer)
2. Build FAISS index with L2 distance
3. For each query image, find K nearest neighbors in the gallery
4. Compute Recall@1, Recall@10, Recall@20

We use a subset (5000 images) for this initial baseline to keep execution fast.

In [11]:
import torch
import torchvision.transforms as T
import torchvision.models as models

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load pretrained ResNet50
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = torch.nn.Identity()  # Remove classification head, keep 2048-dim features
resnet = resnet.to(device)
resnet.eval()

transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f'ResNet50 loaded. Output dim: 2048')

Using device: mps


ResNet50 loaded. Output dim: 2048


In [12]:
# Create a smaller evaluation subset for the baseline
# Use 1000 test products (gallery) + their query images
EVAL_PRODUCTS = 1000

eval_products = gallery_df['product_id'].values[:EVAL_PRODUCTS]
eval_gallery = gallery_df[gallery_df['product_id'].isin(eval_products)].copy().reset_index(drop=True)
eval_query = query_df[query_df['product_id'].isin(eval_products)].copy().reset_index(drop=True)

print(f'Evaluation subset:')
print(f'  Gallery: {len(eval_gallery)} images ({eval_gallery["product_id"].nunique()} products)')
print(f'  Query:   {len(eval_query)} images')

# Collect all indices we need
all_eval_indices = sorted(set(
    eval_gallery['index'].astype(int).tolist() + 
    eval_query['index'].astype(int).tolist()
))
print(f'  Total images to download: {len(all_eval_indices)}')

Evaluation subset:
  Gallery: 1000 images (1000 products)
  Query:   3083 images
  Total images to download: 4083


In [13]:
# Download evaluation images and extract features
ds = load_dataset('Marqo/deepfashion-inshop', split='data', streaming=True)

needed = set(all_eval_indices)
images_by_index = {}

print(f'Downloading {len(needed)} evaluation images...')
for i, ex in enumerate(tqdm(ds, total=max(needed)+1, desc='Downloading')):
    if i in needed:
        img = ex['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        images_by_index[i] = img
        needed.discard(i)
        if not needed:
            break

print(f'Downloaded {len(images_by_index)} images')

Downloading:   0%|          | 0/20035 [00:00<?, ?it/s]

Downloading:   0%|          | 1/20035 [00:19<110:08:13, 19.79s/it]

Downloading:   1%|          | 154/20035 [00:19<30:04, 11.01it/s]  

Downloading:   2%|▏         | 309/20035 [00:19<12:21, 26.61it/s]

Downloading:   2%|▏         | 471/20035 [00:20<06:36, 49.35it/s]

...

Downloading:  98%|█████████▊| 19612/20035 [00:37<00:00, 1552.11it/s]

Downloading:  99%|█████████▊| 19768/20035 [00:37<00:00, 1547.23it/s]

Downloading:  99%|█████████▉| 19932/20035 [00:37<00:00, 1574.53it/s]

Downloading: 100%|█████████▉| 20034/20035 [00:37<00:00, 536.33it/s] 

Downloaded 4083 images


In [14]:
# Extract ResNet50 features
@torch.no_grad()
def extract_features(model, images_dict, indices, transform, device, batch_size=64):
    features = []
    valid_indices = []
    
    batch = []
    batch_idx = []
    
    for idx in tqdm(indices, desc='Extracting features'):
        idx_int = int(idx)
        if idx_int not in images_dict:
            continue
        img = images_dict[idx_int]
        tensor = transform(img)
        batch.append(tensor)
        batch_idx.append(idx_int)
        
        if len(batch) >= batch_size:
            batch_tensor = torch.stack(batch).to(device)
            feats = model(batch_tensor).cpu().numpy()
            features.append(feats)
            valid_indices.extend(batch_idx)
            batch = []
            batch_idx = []
    
    if batch:
        batch_tensor = torch.stack(batch).to(device)
        feats = model(batch_tensor).cpu().numpy()
        features.append(feats)
        valid_indices.extend(batch_idx)
    
    return np.vstack(features), valid_indices

# Extract gallery features
gallery_features, gallery_valid_idx = extract_features(
    resnet, images_by_index, eval_gallery['index'].values, transform, device
)
print(f'Gallery features shape: {gallery_features.shape}')

# Extract query features
query_features, query_valid_idx = extract_features(
    resnet, images_by_index, eval_query['index'].values, transform, device
)
print(f'Query features shape: {query_features.shape}')

Extracting features:   0%|          | 0/1000 [00:00<?, ?it/s]

Extracting features:   6%|▋         | 64/1000 [00:00<00:12, 74.81it/s]

Extracting features:  13%|█▎        | 128/1000 [00:01<00:10, 80.01it/s]

Extracting features:  19%|█▉        | 192/1000 [00:02<00:08, 94.30it/s]

Extracting features:  26%|██▌       | 256/1000 [00:02<00:07, 104.24it/s]

...

Extracting features:  95%|█████████▌| 2944/3083 [00:23<00:01, 123.32it/s]

Extracting features:  98%|█████████▊| 3008/3083 [00:24<00:00, 124.13it/s]

Extracting features:  99%|█████████▉| 3067/3083 [00:24<00:00, 160.13it/s]

Extracting features: 100%|██████████| 3083/3083 [00:24<00:00, 123.71it/s]

Query features shape: (3083, 2048)


In [15]:
# Build FAISS index and evaluate retrieval
import faiss

# L2 normalize for cosine similarity
gallery_normed = gallery_features / np.linalg.norm(gallery_features, axis=1, keepdims=True)
query_normed = query_features / np.linalg.norm(query_features, axis=1, keepdims=True)

# Build index
dim = gallery_normed.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product on normalized vectors = cosine similarity
index.add(gallery_normed.astype(np.float32))
print(f'FAISS index built with {index.ntotal} vectors of dim {dim}')

# Search
K = 20
distances, indices = index.search(query_normed.astype(np.float32), K)

# Map indices to product IDs
gallery_valid_df = eval_gallery[eval_gallery['index'].isin(gallery_valid_idx)].reset_index(drop=True)
query_valid_df = eval_query[eval_query['index'].isin(query_valid_idx)].reset_index(drop=True)

gallery_pids = gallery_valid_df['product_id'].values
query_pids = query_valid_df['product_id'].values

# Compute Recall@K
def compute_recall_at_k(query_pids, gallery_pids, retrieved_indices, k):
    correct = 0
    for i in range(len(query_pids)):
        query_pid = query_pids[i]
        top_k_pids = gallery_pids[retrieved_indices[i, :k]]
        if query_pid in top_k_pids:
            correct += 1
    return correct / len(query_pids)

recall_1 = compute_recall_at_k(query_pids, gallery_pids, indices, 1)
recall_5 = compute_recall_at_k(query_pids, gallery_pids, indices, 5)
recall_10 = compute_recall_at_k(query_pids, gallery_pids, indices, 10)
recall_20 = compute_recall_at_k(query_pids, gallery_pids, indices, 20)

print('\n' + '=' * 50)
print('BASELINE RESULTS: ResNet50 (ImageNet) + FAISS')
print('=' * 50)
print(f'  Recall@1:  {recall_1:.4f} ({recall_1*100:.1f}%)')
print(f'  Recall@5:  {recall_5:.4f} ({recall_5*100:.1f}%)')
print(f'  Recall@10: {recall_10:.4f} ({recall_10*100:.1f}%)')
print(f'  Recall@20: {recall_20:.4f} ({recall_20*100:.1f}%)')

baseline_results = {
    'model': 'ResNet50_ImageNet_V2',
    'embedding_dim': dim,
    'gallery_size': len(gallery_valid_df),
    'query_size': len(query_valid_df),
    'recall_at_1': recall_1,
    'recall_at_5': recall_5,
    'recall_at_10': recall_10,
    'recall_at_20': recall_20,
}

with open(RESULTS / 'metrics.json', 'w') as f:
    json.dump({'phase1_baseline': baseline_results}, f, indent=2)
print(f'\nSaved metrics to results/metrics.json')

FAISS index built with 1000 vectors of dim 2048


## 5. Baseline Analysis
### Per-Category Retrieval Performance

In [ ]:
# Per-category Recall@1 analysis
query_cats = query_valid_df['category2'].values
unique_cats = sorted(set(query_cats))

cat_recalls = {}
for cat in unique_cats:
    mask = query_cats == cat
    if mask.sum() < 5:
        continue
    cat_query_pids = query_pids[mask]
    cat_indices = indices[mask]
    r1 = compute_recall_at_k(cat_query_pids, gallery_pids, cat_indices, 1)
    r10 = compute_recall_at_k(cat_query_pids, gallery_pids, cat_indices, 10)
    cat_recalls[cat] = {'recall@1': r1, 'recall@10': r10, 'n_queries': int(mask.sum())}

print('Per-Category Retrieval Performance (ResNet50 baseline):')
print(f'{"Category":15s} {"R@1":>8s} {"R@10":>8s} {"Queries":>8s}')
print('-' * 42)
for cat in sorted(cat_recalls, key=lambda x: cat_recalls[x]['recall@1'], reverse=True):
    r = cat_recalls[cat]
    print(f'{cat:15s} {r["recall@1"]:8.3f} {r["recall@10"]:8.3f} {r["n_queries"]:8d}')

In [ ]:
# Visualize per-category performance
cats_sorted = sorted(cat_recalls, key=lambda x: cat_recalls[x]['recall@1'], reverse=True)
r1_vals = [cat_recalls[c]['recall@1'] for c in cats_sorted]
r10_vals = [cat_recalls[c]['recall@10'] for c in cats_sorted]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(cats_sorted))
width = 0.35
ax.bar(x - width/2, r1_vals, width, label='Recall@1', color='#3498db', alpha=0.8)
ax.bar(x + width/2, r10_vals, width, label='Recall@10', color='#e74c3c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(cats_sorted, rotation=45, ha='right')
ax.set_ylabel('Recall')
ax.set_title('Per-Category Retrieval Performance (ResNet50 Baseline)', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(recall_1, color='#3498db', linestyle='--', alpha=0.5, label=f'Overall R@1={recall_1:.3f}')
ax.axhline(recall_10, color='#e74c3c', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(RESULTS / 'baseline_per_category.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/baseline_per_category.png')

### Retrieval Examples: Successes and Failures

In [ ]:
# Show retrieval examples: 3 successes and 3 failures
successes = []
failures = []

for i in range(len(query_pids)):
    is_correct = query_pids[i] == gallery_pids[indices[i, 0]]
    if is_correct and len(successes) < 3:
        successes.append(i)
    elif not is_correct and len(failures) < 3:
        failures.append(i)
    if len(successes) >= 3 and len(failures) >= 3:
        break

def show_retrieval(query_idx, retrieved_indices, title_prefix, gallery_imgs, query_imgs):
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    fig.suptitle(title_prefix, fontsize=12, fontweight='bold')
    
    q_idx = int(query_valid_df.iloc[query_idx]['index'])
    if q_idx in images_by_index:
        axes[0].imshow(images_by_index[q_idx])
    axes[0].set_title(f'QUERY\n{query_valid_df.iloc[query_idx]["category2"]}', fontsize=9, color='blue')
    axes[0].axis('off')
    
    for j in range(5):
        gal_pos = retrieved_indices[query_idx, j]
        g_idx = int(gallery_valid_df.iloc[gal_pos]['index'])
        is_match = query_pids[query_idx] == gallery_pids[gal_pos]
        
        if g_idx in images_by_index:
            axes[j+1].imshow(images_by_index[g_idx])
        
        color = 'green' if is_match else 'red'
        mark = 'MATCH' if is_match else 'wrong'
        axes[j+1].set_title(f'#{j+1} ({mark})\n{gallery_valid_df.iloc[gal_pos]["category2"]}\nsim={distances[query_idx,j]:.3f}', 
                           fontsize=8, color=color)
        axes[j+1].axis('off')
    
    plt.tight_layout()
    return fig

print('=== SUCCESSFUL RETRIEVALS ===')
for i, s_idx in enumerate(successes):
    fig = show_retrieval(s_idx, indices, f'Success #{i+1}: Query → Top-5 Retrieved', images_by_index, images_by_index)
    plt.show()

print('\n=== FAILED RETRIEVALS ===')
for i, f_idx in enumerate(failures):
    fig = show_retrieval(f_idx, indices, f'Failure #{i+1}: Query → Top-5 Retrieved', images_by_index, images_by_index)
    plt.show()

plt.savefig(RESULTS / 'retrieval_examples.png', dpi=150, bbox_inches='tight')
print('\nSaved: results/retrieval_examples.png')

## 6. Similarity Distribution Analysis

In [ ]:
# Distribution of cosine similarities for correct vs incorrect matches
correct_sims = []
incorrect_sims = []

for i in range(len(query_pids)):
    for j in range(K):
        sim = distances[i, j]
        if query_pids[i] == gallery_pids[indices[i, j]]:
            correct_sims.append(sim)
        else:
            incorrect_sims.append(sim)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(correct_sims, bins=50, alpha=0.6, label=f'Correct matches (n={len(correct_sims)})', color='green', density=True)
ax.hist(incorrect_sims, bins=50, alpha=0.6, label=f'Incorrect matches (n={len(incorrect_sims)})', color='red', density=True)
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Density')
ax.set_title('Similarity Score Distribution: Correct vs Incorrect Matches', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(RESULTS / 'similarity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Correct matches - mean sim: {np.mean(correct_sims):.4f}, std: {np.std(correct_sims):.4f}')
print(f'Incorrect matches - mean sim: {np.mean(incorrect_sims):.4f}, std: {np.std(incorrect_sims):.4f}')
print(f'Separation (mean diff): {np.mean(correct_sims) - np.mean(incorrect_sims):.4f}')
print(f'\nThis shows {"good" if np.mean(correct_sims) - np.mean(incorrect_sims) > 0.05 else "poor"} '
      f'separation between correct and incorrect matches.')
print('Saved: results/similarity_distribution.png')

## 7. Summary & Next Steps

In [ ]:
print('=' * 60)
print('PHASE 1 SUMMARY')
print('=' * 60)

print(f'''
Dataset: DeepFashion In-Shop (Marqo/deepfashion-inshop)
  - {n_images:,} images, {n_products:,} products, {n_categories} categories
  - Rich metadata: gender, category, color, text descriptions
  - ~4 views per product (front, side, back, full)

Primary Metric: Recall@K (standard for image retrieval)

Baseline: ResNet50 (ImageNet V2 weights) + FAISS cosine similarity
  - Recall@1:  {recall_1:.4f} ({recall_1*100:.1f}%)
  - Recall@5:  {recall_5:.4f} ({recall_5*100:.1f}%)
  - Recall@10: {recall_10:.4f} ({recall_10*100:.1f}%)
  - Recall@20: {recall_20:.4f} ({recall_20*100:.1f}%)

Published SOTA comparison:
  - FashionNet (2016):    R@1 = 53.0%
  - CLIP ViT-B/32 (2021): R@1 ~ 78%
  - DINOv2 ViT-B (2023):  R@1 ~ 82%

Key observations:
  1. ResNet50 without fine-tuning provides a {"reasonable" if recall_1 > 0.2 else "weak"} baseline
  2. Dataset is heavily skewed: women (85%) vs men (15%)
  3. Category imbalance: tees (27%) dominate; suiting (0.07%) is rare
  4. Multiple views per product enable robust retrieval evaluation

Next steps (Phase 2):
  - Compare CLIP, DINOv2, EfficientNet, ViT against ResNet50 baseline
  - Test different embedding dimensions (512 vs 768 vs 1024 vs 2048)
  - Evaluate FAISS index types (Flat vs HNSW vs IVF)
''')